In [ ]:
!pip install torch torchvision pillow


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import zipfile
import os

# Path to the ZIP file on Google Drive
zip_file_path = '/content/drive/MyDrive/archive (2).zip'  # Update this with the path to your ZIP file

# Directory where you want to extract the ZIP file
extract_dir = '/content/drive/MyDrive/extracted_images'  # You can change this path

# Extract the ZIP file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Dataset extracted to: {extract_dir}")


Dataset extracted to: /content/drive/MyDrive/extracted_images


In [ ]:
import os

# List the files in the extracted directory
extracted_files = os.listdir(extract_dir)
print(f"Files in extracted directory: {extracted_files[:10]}")  # Show first 10 files


Files in extracted directory: ['k.jpg', 'old_photo_01.jpg', 'old_photo_02.jpg', 'old_photo_03.jpg', 'old_photo_04.jpg', 'old_photo_05.jpg', 'old_photo_06.jpg', 'old_photo_07.jpg', 'old_photo_08.jpeg', 'old_photo_09.jpeg']


In [ ]:
# 1. Import Required Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

# 2. Define the Model (U-Net)
class UNetColorization(nn.Module):
    def __init__(self):
        super(UNetColorization, self).__init__()

        # Encoding layers (downsampling)
        self.enc1 = self.conv_block(3, 64)
        self.enc2 = self.conv_block(64, 128)
        self.enc3 = self.conv_block(128, 256)
        self.enc4 = self.conv_block(256, 512)

        # Bottleneck (bottom of the U)
        self.bottleneck = self.conv_block(512, 1024)

        # Decoding layers (upsampling)
        self.dec4 = self.conv_block(1024, 512)
        self.dec3 = self.conv_block(512, 256)
        self.dec2 = self.conv_block(256, 128)
        self.dec1 = self.conv_block(128, 64)

        # Final output layer
        self.out_conv = nn.Conv2d(64, 3, kernel_size=1)

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

    def upconv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2),
            nn.ReLU()
        )

    def forward(self, x):
        # Encoding
        enc1 = self.enc1(x)
        enc2 = self.enc2(enc1)
        enc3 = self.enc3(enc2)
        enc4 = self.enc4(enc3)

        # Bottleneck
        bottleneck = self.bottleneck(enc4)

        # Decoding
        dec4 = self.upconv_block(bottleneck, 512)
        dec3 = self.upconv_block(dec4, 256)
        dec2 = self.upconv_block(dec3, 128)
        dec1 = self.upconv_block(dec2, 64)

        # Output layer
        out = self.out_conv(dec1)
        return out


# 3. Custom Dataset to Load Grayscale and Color Images
class ColorizationDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.image_names = os.listdir(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = os.path.join(self.image_dir, self.image_names[idx])
        img = Image.open(img_name).convert('RGB')  # Ensure color images are RGB

        # Convert to grayscale and then to 3-channel (RGB)
        grayscale_img = img.convert('L').convert('RGB')  # Convert grayscale to RGB (3 channels)

        # Apply transformation if provided
        if self.transform:
            img = self.transform(img)   # Color image (RGB)
            grayscale_img = self.transform(grayscale_img)  # Grayscale image converted to 3-channel RGB

        # Return both grayscale image and color image
        return grayscale_img, img


# 4. Set Up Image Transformations
transform = transforms.Compose([
    transforms.ToTensor(),           # Convert to tensor
    transforms.Resize((256, 256))    # Resize all images to 256x256
])

# 5. Initialize DataLoader
image_dir = '/content/drive/MyDrive/extracted_images/'  # Update with your path
dataset = ColorizationDataset(image_dir=image_dir, transform=transform)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

# 6. Initialize Model, Loss Function, Optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # Check for GPU availability
model = UNetColorization().to(device)  # Move model to GPU if available
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 7. Training Loop
num_epochs = 10

for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0

    # Iterate over batches of the dataset
    for batch_idx, (grayscale, color) in enumerate(train_loader):
        grayscale = grayscale.to(device)  # Move data to GPU
        color = color.to(device)

        # Forward pass
        output = model(grayscale)

        # Compute loss
        loss = criterion(output, color)

        # Backward pass and optimize
        optimizer.zero_grad()  # Zero gradients before backward pass
        loss.backward()        # Backpropagate the loss
        optimizer.step()       # Update the model parameters

        running_loss += loss.item()

    # Print the loss for this epoch
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader)}")


RuntimeError: Boolean value of Tensor with more than one value is ambiguous